# Imports

In [1]:
import os
import cv2
import subprocess
import pandas as pd
from google.cloud import storage
from collections import defaultdict
from IPython.display import Video, display
from google.colab.patches import cv2_imshow

# Set Up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install mediapipe
!wget -q -O efficientdet.tflite -q https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/int8/1/efficientdet_lite0.tflite

In [ ]:
!gcloud auth login # Authenticates your identity for general gcloud CLI command use e.g. gcloud storage cp, gcloud compute, gsutil
!gcloud auth application-default login # Authenticates your environment for API access e.g., storage.Client()

In [ ]:
client = storage.Client(project="brb-traffic")

In [6]:
base_dir = "/"
bucket_name = 'brb-traffic'
video_path='videos'
os.makedirs(video_path, exist_ok=True)

# Read Data

In [ ]:
# Load the dataset
train = pd.read_csv(os.path.join(base_dir, '/content/Train.csv'))

# Extract camera ID and reconstruct the video path
def extract_camera_path(video):
    parts = video.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[0]
    return f"{camera_id}/{filename}"

train['videos'] = train['videos'].apply(extract_camera_path)

# Display shape and preview
display(train.shape, train.head())

In [ ]:
ss = pd.read_csv(os.path.join(base_dir,'/content/SampleSubmission.csv'))
display(ss.shape,ss.head())

In [ ]:
# test = pd.read_csv(os.path.join(base_dir,'TestInputSegments.csv'))
# display(test.shape,test.head())

# Download files from a google cloud storage

In [ ]:
folder_counts = defaultdict(int)
folder_sizes = defaultdict(int)
largest_file = ("", 0)  # (name, size)

for blob_name in blobs: # Iterate over the list of blob names (strings)
    blob_obj = client.bucket(bucket_name).blob(blob_name) # Get the actual blob object

    # Ensure the blob exists before proceeding
    if blob_obj.exists():
        folder = blob_obj.name.split('/')[0] if '/' in blob_obj.name else '(root)'

        # Update counts
        folder_counts[folder] += 1

        # Update total sizes
        folder_sizes[folder] += blob_obj.size or 0  # Use 0 if blob_obj.size is None

        # Check largest file
        if blob_obj.size is not None and blob_obj.size > largest_file[1]: # Add check for None
            largest_file = (blob_obj.name, blob_obj.size)
    else:
        print(f"Warning: Blob '{blob_name}' not found in bucket '{bucket_name}'. Skipping.")


# Print per-folder stats
print("Folder stats:")
for folder in folder_counts:
    size_mb = folder_sizes[folder] / (1024*1024)
    print(f"{folder}: {folder_counts[folder]} files, {size_mb:.2f} MB")

# Print largest file info
largest_mb = largest_file[1] / (1024*1024)
print(f"\nLargest file in bucket: {largest_file[0]} ({largest_mb:.2f} MB)")

In [ ]:
blobs=train.videos.tolist()[:2]
blobs

In [ ]:
for blob_name in blobs:
    blob = client.bucket(bucket_name).blob(blob_name)
    file_name = os.path.basename(blob_name)  # get last part after '/'
    local_path = os.path.join(video_path, file_name)

    print("Downloading:", blob_name)
    blob.download_to_filename(local_path)
    print("✅ Downloaded to:", local_path)

# Play some videos downloaded earlier

In [13]:
def show_video(video_path, trim=False, duration=10, width=800):
    """
    Convert (and optionally trim) a video, suppress ffmpeg output, and display it inline.
    Parameters:
        video_path (str): Path to the input video file.
        trim (bool): Whether to trim the video to a short preview (default False).
        duration (int): Duration in seconds if trimming (default 10).
        width (int): Display width in pixels (default 800).
    """
    output_path = "preview.mp4"

    # Build ffmpeg command
    cmd = ["ffmpeg", "-i", video_path]
    if trim:
        cmd += ["-t", str(duration)]
    cmd += [output_path, "-y"]  # overwrite existing

    # Run ffmpeg silently (no stdout/stderr)
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Display the video inline
    display(Video(output_path, width=width, embed=True))

In [ ]:
counter = 0
for video in os.listdir(video_path):
    full_path = os.path.join(video_path, video)
    show_video(full_path)  # display the full video
    #show_video("normanniles1_2025-10-20-06-01-45.mp4", trim=True)     # 10s preview
    #show_video("normanniles1_2025-10-20-06-01-45.mp4", trim=True, duration=5)  # 5s preview
    counter += 1
    if counter == 2:
        break

# Mediapipe setup

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import os
from google.colab.patches import cv2_imshow


model_path = '/absolute/path/to/lite-model_efficientdet_lite0_detection_metadata_1.tflite'


BaseOptions = mp.tasks.BaseOptions
ObjectDetector = mp.tasks.vision.ObjectDetector
ObjectDetectorOptions = mp.tasks.vision.ObjectDetectorOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = ObjectDetectorOptions(
    base_options=BaseOptions(model_asset_path='/content/efficientdet.tflite'),
    max_results=5,
    running_mode=VisionRunningMode.VIDEO)

with ObjectDetector.create_from_options(options) as detector:
    for video_file_name in os.listdir(video_path):
        full_path = os.path.join(video_path, video_file_name)
        cap = cv2.VideoCapture(full_path)
        video_file_fps = cap.get(cv2.CAP_PROP_FPS)

        frame_index = 0
        max_frames_to_process = 100 # Limit frames for demonstration/debugging

        if not cap.isOpened():
            print(f"Error: Could not open video file {full_path}")
            continue

        print(f"Processing video: {video_file_name}")
        while True: # Loop until break
            ret, frame = cap.read()
            if not ret or frame_index >= max_frames_to_process:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            frame_timestamp_ms = int(1000 * frame_index / video_file_fps) if video_file_fps > 0 else 0

            detection_result = detector.detect_for_video(mp_image, frame_timestamp_ms)

            # Draw detections on the frame
            annotated_frame = frame.copy() # Make a copy to draw on
            for detection in detection_result.detections:
                bbox = detection.bounding_box
                start_point = (bbox.origin_x, bbox.origin_y)
                end_point = (bbox.origin_x + bbox.width, bbox.origin_y + bbox.height)
                cv2.rectangle(annotated_frame, start_point, end_point, (0, 255, 0), 2) # Green rectangle

                category_name = detection.categories[0].category_name if detection.categories else "Unknown"
                score = round(detection.categories[0].score, 2) if detection.categories else 0.0
                text = f"{category_name}: {score}"
                cv2.putText(annotated_frame, text, (bbox.origin_x, bbox.origin_y - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            cv2_imshow(annotated_frame) # Display the annotated frame

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            frame_index += 1

        cap.release() # Release the capture object after processing each video
        cv2.destroyAllWindows() # Close any OpenCV windows (important for local execution environments)
